# Notebook 06 — AlphaMissense Subset for KO4

KO4 (`2_Analyses/KO4_domain_AM_3D.ipynb`) needs the AlphaMissense score of **every possible** missense variant in four driver proteins (KRAS, TP53, SMAD4, CDKN2A) to compute the mean pathogenicity per residue. The full AlphaMissense hg38 table is 564 MB, too large for the repository, so KO4 reads a small extract that is shipped in `3_Data/`.

**Shipped data vs. this notebook**

- **Source:** `alphamissense_hg38.parquet` — a parquet conversion of `AlphaMissense_hg38.tsv.gz` ([Cheng et al. 2023](https://doi.org/10.1126/science.adg7492); original release: [zenodo.org/records/10813168](https://zenodo.org/records/10813168), CC BY 4.0). 71,697,556 rows; MD5 `498ea7c4778906ca42ad2d1a1a33e747`. Download it from this paper's Zenodo record ([doi.org/10.5281/zenodo.24132119](https://doi.org/10.5281/zenodo.24132119)) into `3_Data/` (git-ignored), or set the environment variable `AM_FULL_PARQUET` to its location.
- **Output:** `3_Data/alphamissense_hg38_KO4_drivers.parquet` — every row of the source table for the four UniProt IDs, all columns, in the original row order. No rows are de-duplicated: a protein variant can appear several times (different nucleotide changes), and KO4 averages over all of them.

The filter is deterministic, so running this notebook regenerates the shipped extract with identical contents.

## Output schema — `alphamissense_hg38_KO4_drivers.parquet`

Same columns as the source table. One row per genomic single-nucleotide missense variant.

| Column | Type | Notes |
|--------|------|-------|
| `chrom`, `pos`, `ref`, `alt` | String, Int64, String, String | hg38 coordinates and alleles |
| `genome` | String | `hg38` |
| `uniprot_id` | String | P01116 (KRAS), P04637 (TP53), Q13485 (SMAD4), P42771 (CDKN2A) |
| `transcript_id` | String | Ensembl canonical transcript |
| `protein_variant` | String | Amino acid substitution (e.g. `G12D`) |
| `am_pathogenicity` | Float64 | AlphaMissense score 0–1 |
| `am_class` | String | `likely_benign` / `ambiguous` / `likely_pathogenic` |

---
## 1 — Setup

In [1]:
import hashlib
import os
from pathlib import Path

import polars as pl

DATA_DIR    = Path('../3_Data')
AM_FULL     = Path(os.environ.get('AM_FULL_PARQUET', DATA_DIR / 'alphamissense_hg38.parquet'))
AM_FULL_MD5 = '498ea7c4778906ca42ad2d1a1a33e747'
MERGED      = DATA_DIR / 'merged_tcga_1000G.parquet'
OUT_PARQUET = DATA_DIR / 'alphamissense_hg38_KO4_drivers.parquet'

# KO4 driver proteins (same UniProt IDs as PROTEINS in KO4)
DRIVER_UNIPROT = {'KRAS': 'P01116', 'TP53': 'P04637', 'SMAD4': 'Q13485', 'CDKN2A': 'P42771'}

pl.Config.set_tbl_rows(-1)


def md5(path: Path) -> str:
    h = hashlib.md5()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(1 << 24), b''):
            h.update(block)
    return h.hexdigest()

---
## 2 — Checks

- The source file is the exact table used for the reported results (MD5).
- The UniProt IDs match those the merged dataset assigns to the four genes (KO4 uses both).

In [2]:
full_md5 = md5(AM_FULL)
assert full_md5 == AM_FULL_MD5, f'Unexpected AlphaMissense source file: MD5 {full_md5}'
print(f'Source MD5 : {full_md5}  (OK)')

merged_ids = (
    pl.read_parquet(MERGED, columns=['gene_name', 'uniprot_id'])
    .filter(pl.col('gene_name').is_in(list(DRIVER_UNIPROT)))
    .drop_nulls()
    .unique()
    .sort('gene_name')
)
print(merged_ids)
assert dict(merged_ids.iter_rows()) == DRIVER_UNIPROT, 'Gene → UniProt mapping differs from merged dataset'

Source MD5 : 498ea7c4778906ca42ad2d1a1a33e747  (OK)


shape: (4, 2)
┌───────────┬────────────┐
│ gene_name ┆ uniprot_id │
│ ---       ┆ ---        │
│ str       ┆ str        │
╞═══════════╪════════════╡
│ CDKN2A    ┆ P42771     │
│ KRAS      ┆ P01116     │
│ SMAD4     ┆ Q13485     │
│ TP53      ┆ P04637     │
└───────────┴────────────┘


---
## 3 — Extract

In [3]:
df_sub = (
    pl.scan_parquet(AM_FULL)
    .filter(pl.col('uniprot_id').is_in(list(DRIVER_UNIPROT.values())))
    .collect()
)

summary = (
    df_sub
    .with_columns(
        pl.col('protein_variant').str.extract(r'(\d+)', 1).cast(pl.Int32).alias('residue')
    )
    .group_by('uniprot_id')
    .agg(
        pl.len().alias('rows'),
        pl.col('protein_variant').n_unique().alias('protein_variants'),
        pl.col('residue').n_unique().alias('residues'),
        pl.col('residue').max().alias('max_residue'),
    )
    .sort('uniprot_id')
)
print(summary)
print(f'Total rows: {df_sub.height:,}')

shape: (4, 5)
┌────────────┬──────┬──────────────────┬──────────┬─────────────┐
│ uniprot_id ┆ rows ┆ protein_variants ┆ residues ┆ max_residue │
│ ---        ┆ ---  ┆ ---              ┆ ---      ┆ ---         │
│ str        ┆ u32  ┆ u32              ┆ u32      ┆ i32         │
╞════════════╪══════╪══════════════════╪══════════╪═════════════╡
│ P01116     ┆ 1253 ┆ 1110             ┆ 188      ┆ 189         │
│ P04637     ┆ 2569 ┆ 2314             ┆ 392      ┆ 393         │
│ P42771     ┆ 983  ┆ 904              ┆ 155      ┆ 156         │
│ Q13485     ┆ 3631 ┆ 3245             ┆ 551      ┆ 552         │
└────────────┴──────┴──────────────────┴──────────┴─────────────┘
Total rows: 8,436


---
## 4 — Save

In [4]:
df_sub.write_parquet(OUT_PARQUET, compression='zstd', compression_level=19)
print(f'Saved: {OUT_PARQUET}  ({OUT_PARQUET.stat().st_size / 2**10:.0f} KB, MD5 {md5(OUT_PARQUET)})')

# Round trip: the saved file holds exactly the extracted rows, in order
assert pl.read_parquet(OUT_PARQUET).equals(df_sub)

Saved: ..\3_Data\alphamissense_hg38_KO4_drivers.parquet  (53 KB, MD5 41584e689e3d78e27f333f32dda2f634)
